In [ ]:
%pip install opencv-python imageio mediapipe kagglehub

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
# import torchvision

import cv2
# import tensorflow as tf

import numpy as np
from typing import List
from matplotlib import pyplot as plt
import imageio
from IPython.display import Image, display

from torch.utils.data import DataLoader, Dataset

from aligned_dataframe import load_aligned_dataframe, load_word_frames
from models.valente.model import LipNet, ctc_greedy_decode, tokens_to_text
from processing.preprocessor import Preprocessor
from processing.video_loader import VideoLoader


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


# building data loading functions

download dataset from kaggle
https://www.kaggle.com/datasets/mohamedbentalb/lipreading-dataset and save them in data\lips-dataset

In [ ]:
%pip install -q kagglehub

import shutil
from pathlib import Path
import kagglehub

DATASET_HANDLE = 'mohamedbentalb/lipreading-dataset'
target_dir = Path('data/lips-dataset')

base = Path(kagglehub.dataset_download(DATASET_HANDLE))
for candidate in (base, base / 'data'):
    if (candidate / 's1').is_dir() or (candidate / 'alignments').is_dir():
        base = candidate
        break

print(f'Downloaded to: {base}')

target_dir.mkdir(parents=True, exist_ok=True)
for name in ('s1', 'alignments'):
    src = base / name
    dst = target_dir / name
    if not src.exists():
        print(f'SKIP: {name} not found in {base}')
    elif dst.exists():
        print(f'SKIP: {name} already at {dst}')
    else:
        shutil.move(str(src), str(dst))
        print(f'Moved {name} -> {dst}')


#### probably inside LBY data cleaning - to be deleted or remove. 

In [ ]:
# # test_path = r'.\data\lips-dataset\s1\bbaf2n.mpg' ## Windows File Directory
# # test_path = r'./data/lips-dataset/s1/bbaf2n.mpg' ## Macbook File Directory
# test_path = os.path.join("data", "lips-dataset", "s1", "bbaf2n.mpg") # testing for all directory
# video_loader = VideoLoader()
# video_data = video_loader.load(test_path)
# # video_data

In [ ]:
# preprocessor = Preprocessor()
# video_data = preprocessor.convert_to_greyscale(video_data)

if fail to crop lips, head over to google https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task



In [ ]:
# vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

In [ ]:
# char_to_num = {char: idx for idx, char in enumerate(vocab)}
# num_to_char = {idx: char for char, idx in char_to_num.items()}

# print(f"The vocabulary is: {vocab} (size = {len(vocab)})")

In [ ]:
# def encode_text(text):
#     return torch.tensor([char_to_num[c] for c in text], dtype=torch.long)

# def decode_indices(indices):
#     return "".join(num_to_char[int(i)] for i in indices)

# x = encode_text("hello")
# print(x)
# print(decode_indices(x))

In [ ]:
# def load_alignments(path:str) -> List[str]: 
#     with open(path, 'r') as f: 
#         lines = f.readlines() 
#     tokens = []
#     for line in lines:
#         line = line.split()
#         if line[2] != 'sil': 
#             tokens = [*tokens,' ',line[2]]
#     return [encode_text(x) for x in tokens]

# def load_data(path: str) -> dict: 
#     # path = bytes.decode(path.numpy())
#     #file_name = path.split('/')[-1].split('.')[0]
#     # File name splitting for windows
#     file_name = os.path.basename(path).split('.')[0]
#     video_path = os.path.join('data','lips-dataset','s1',f'{file_name}.mpg')
#     alignment_path = os.path.join('data', 'lips-dataset', 'alignments','s1',f'{file_name}.align')
#     video_data = video_loader.load(path) 
#     alignments = load_alignments(alignment_path)
#     video_data['alignments'] = alignments
#     return video_data


# video_data = load_data(test_path)
# # print(video_data)
# print(video_data['alignments'])
# print([decode_indices(x) for x in video_data['alignments']])

# Creating data pipeline

## Alignment Algorithm
### LBY: The cell below runs the align_videos.py as if it was in a terminal, input arguments include data directory and output directory
### It is important to note to save the kaggle dataset in its subfolder called data/lips-dataset, of which should have the following subfolders:
### (a) alignments, (b) s1, (c) processedData [This is an addition]

In [ ]:
!python align_videos.py --video-dir data/lips-dataset/s1 --align-dir data/lips-dataset/alignments/s1 --output-dir data/lips-dataset/processedData --grayscale --crop-mouth

## Train/Test Split & Model Evaluation
### LBY: Split `aligned_words.csv` (word-level, stratified 80/20), train a LipNet-style classifier, and report test accuracy.


In [ ]:
from dataset import get_word_classification_dataloaders
from models.LBY.model import LipReadingClassifier

train_loader, test_loader, word_to_index, index_to_word = get_word_classification_dataloaders(
    test_size=0.2, seed=42, batch_size=32, stratify=True
)
num_classes = len(word_to_index)

print(f'Device: {device}')
print(f'Train samples: {len(train_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Classes: {num_classes}')


In [ ]:
# train_loader.dataset.dataframe

In [ ]:
# test_loader.dataset.dataframe

In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

from models.LBY.train import set_seed, train_one_epoch, evaluate

set_seed(42)

model = LipReadingClassifier(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
out_dir = Path('checkpoints')
out_dir.mkdir(parents=True, exist_ok=True)

epochs = 100
best_accuracy = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(
        f'Epoch {epoch:2d}/{epochs} | '
        f'train_loss {train_loss:.4f} train_acc {train_acc:.4f} | '
        f'test_loss {test_loss:.4f} test_acc {test_acc:.4f}'
    )

    if test_acc > best_accuracy:
        best_accuracy = test_acc
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'word_to_index': word_to_index,
            'test_accuracy': test_acc,
        }
        torch.save(checkpoint, out_dir / 'best.pt')

print(f'Best test accuracy: {best_accuracy:.4f}')

In [ ]:
# Optionally load the checkpoint saved above and re-evaluate.
checkpoint = torch.load('checkpoints/best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
saved_epoch = checkpoint['epoch']
saved_acc = checkpoint['test_accuracy']
print(f'Loaded checkpoint (epoch {saved_epoch}, saved test acc {saved_acc:.4f})')
print(f'Re-evaluated test accuracy: {evaluate(model, test_loader, criterion, device)[1]:.4f}')


## Testing with sample video

### Valente: predicting on a sample video. via the best model

In [ ]:
test_path = os.path.join("data", "lips-dataset", "s1", "swwv7s.mpg") # testing for all directory


In [ ]:
# 1. Load .mpg
video_loader = VideoLoader()
preprocessor = Preprocessor()
video_frames = video_loader.load(test_path)
preprocessed_frames = preprocessor.convert_to_greyscale(video_frames)

print(type(preprocessed_frames['frames']))

In [ ]:
# 3. Convert to tensor
batch = torch.from_numpy(preprocessed_frames['frames'].astype(np.float32)).unsqueeze(0).unsqueeze(2)  # → (1, T, 1, 50, 100)
# 4. Inference
model.eval()
with torch.no_grad():
    logits = model(batch.to(device))
    pred_idx = logits.argmax(dim=1).item()
    pred_word = index_to_word[pred_idx]

print(f"Predicted word: {pred_word}")

In [ ]:
test_path = np.load(os.path.join("data", "lips-dataset", "processedData", "swwv9a.npz"))['frames']

In [ ]:
# 3. Convert to tensor
batch = torch.from_numpy(test_path.astype(np.float32)).unsqueeze(0).unsqueeze(2)  # → (1, T, 1, 50, 100)
# 4. Inference
model.eval()
with torch.no_grad():
    logits = model(batch.to(device))
    pred_idx = logits.argmax(dim=1).item()
    pred_word = index_to_word[pred_idx]

print(f"Predicted word: {pred_word}")

# Train/test split & model evaluation

In [ ]:
# LipNet uses character-level CTC. Index 0 is reserved for the CTC blank.
recommended_epochs = 100
recommended_learning_rate = 1e-4
recommended_batch_size = 4
seed = 42

dataframe = load_aligned_dataframe()
characters = sorted(set(' '.join(dataframe['word'].astype(str))))
char_to_index = {char: index + 1 for index, char in enumerate(characters)}
index_to_char = {index: char for char, index in char_to_index.items()}
blank_index = 0
num_classes = len(char_to_index) + 1

In [ ]:
class SentenceDataset(Dataset):
    def __init__(self, manifest, video_ids):
        self.items = []
        for video_id in video_ids:
            rows = manifest[manifest['video_id'] == video_id].sort_values('word_index')
            text = ' '.join(rows['word'].astype(str))
            self.items.append((str(video_id), rows, text))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        video_id, rows, text = self.items[index]
        frames = np.concatenate([load_word_frames(row) for _, row in rows.iterrows()], axis=0)
        frames = torch.from_numpy(frames.astype(np.float32) / 255.0)
        frames = (frames - frames.mean()) / (frames.std() + 1e-6)
        frames = frames.unsqueeze(1)
        targets = torch.tensor([char_to_index[char] for char in text], dtype=torch.long)
        return frames, targets, text, video_id

def sentence_collate(batch):
    frames, targets, texts, video_ids = zip(*batch)
    input_lengths = torch.tensor([item.size(0) for item in frames], dtype=torch.long)
    target_lengths = torch.tensor([item.size(0) for item in targets], dtype=torch.long)
    padded_frames = torch.zeros(len(frames), int(input_lengths.max()), *frames[0].shape[1:])
    for index, item in enumerate(frames):
        padded_frames[index, :item.size(0)] = item
    return padded_frames, torch.cat(targets), input_lengths, target_lengths, texts, video_ids

video_ids = np.array(sorted(dataframe['video_id'].dropna().unique()))
rng = np.random.default_rng(seed)
rng.shuffle(video_ids)
split_index = int(len(video_ids) * 0.8)
train_dataset = SentenceDataset(dataframe, video_ids[:split_index])
test_dataset = SentenceDataset(dataframe, video_ids[split_index:])
train_loader = DataLoader(train_dataset, batch_size=recommended_batch_size, shuffle=True, collate_fn=sentence_collate)
test_loader = DataLoader(test_dataset, batch_size=recommended_batch_size, shuffle=False, collate_fn=sentence_collate)

print(f'Train sentences: {len(train_dataset)} | Test sentences: {len(test_dataset)}')
print(f'Character vocabulary ({len(char_to_index)}): {characters}')

In [ ]:
model = LipNet(num_classes=num_classes, input_channels=1).to(device)
ctc_loss = nn.CTCLoss(blank=blank_index, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=recommended_learning_rate)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 85], gamma=0.1)
model

In [ ]:
def decode_log_probs(log_probs, input_lengths):
    token_sequences = ctc_greedy_decode(log_probs.cpu(), input_lengths.cpu(), blank_index)
    return [tokens_to_text(tokens, index_to_char) for tokens in token_sequences]

def character_error_rate(prediction, target):
    previous = list(range(len(target) + 1))
    for row, predicted_char in enumerate(prediction, start=1):
        current = [row]
        for column, target_char in enumerate(target, start=1):
            current.append(min(previous[column] + 1, current[column - 1] + 1, previous[column - 1] + (predicted_char != target_char)))
        previous = current
    return previous[-1] / max(len(target), 1)

def run_epoch(loader, training):
    model.train(training)
    total_loss = total_cer = 0.0
    total_exact = total_sentences = 0
    for batch_frames, batch_targets, input_lengths, target_lengths, texts, _ in loader:
        batch_frames = batch_frames.to(device)
        batch_targets = batch_targets.to(device)
        with torch.set_grad_enabled(training):
            log_probs, output_lengths = model.ctc_log_probs(batch_frames, input_lengths)
            loss = ctc_loss(log_probs, batch_targets, output_lengths.cpu(), target_lengths.cpu())
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
        predictions = decode_log_probs(log_probs.detach(), output_lengths)
        total_loss += loss.item() * len(texts)
        total_cer += sum(character_error_rate(prediction, target) for prediction, target in zip(predictions, texts))
        total_exact += sum(prediction == target for prediction, target in zip(predictions, texts))
        total_sentences += len(texts)
    return total_loss / total_sentences, total_cer / total_sentences, total_exact / total_sentences

@torch.no_grad()
def preview_predictions(loader, limit=3):
    model.eval()
    batch_frames, _, input_lengths, _, texts, _ = next(iter(loader))
    log_probs, output_lengths = model.ctc_log_probs(batch_frames.to(device), input_lengths)
    for target, prediction in zip(texts[:limit], decode_log_probs(log_probs, output_lengths)[:limit]):
        print('Target:    ', target)
        print('Prediction:', prediction)
        print('~' * 80)

In [ ]:
epochs = recommended_epochs
checkpoint_path = Path('checkpoints/valente_lipnet_ctc.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
best_cer = float('inf')

for epoch in range(1, epochs + 1):
    train_loss, train_cer, train_exact = run_epoch(train_loader, training=True)
    test_loss, test_cer, test_exact = run_epoch(test_loader, training=False)
    scheduler.step()
    print(f'Epoch {epoch:03d}/{epochs} | train loss {train_loss:.4f} CER {train_cer:.3f} exact {train_exact:.3f} | test loss {test_loss:.4f} CER {test_cer:.3f} exact {test_exact:.3f} | lr {optimizer.param_groups[0]["lr"]:.1e}')
    preview_predictions(test_loader)
    if test_cer < best_cer:
        best_cer = test_cer
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'char_to_index': char_to_index, 'blank_index': blank_index, 'test_cer': test_cer}, checkpoint_path)

print('Best test CER:', best_cer)
print('Saved checkpoint:', checkpoint_path)

### Loading checkpoints

In [ ]:
checkpoint = torch.load('checkpoints/valente_lipnet_ctc.pt', map_location=device)
loaded_char_to_index = checkpoint['char_to_index']
loaded_index_to_char = {index: char for char, index in loaded_char_to_index.items()}
loaded_blank_index = checkpoint['blank_index']
inference_model = LipNet(num_classes=len(loaded_char_to_index) + 1, input_channels=1).to(device)
inference_model.load_state_dict(checkpoint['model_state_dict'])
inference_model.eval()
print('Loaded epoch:', checkpoint['epoch'], '| Test CER:', checkpoint['test_cer'])

### running predictions

In [ ]:
@torch.no_grad()
def predict_video(video_path):
    loader = VideoLoader()
    preprocessor = Preprocessor()
    video_data = preprocessor.convert_to_greyscale(loader.load(str(video_path)))
    frames_array = np.asarray(video_data['frames'], dtype=np.uint8)
    frames_tensor = torch.from_numpy(frames_array.astype(np.float32) / 255.0)
    frames_tensor = (frames_tensor - frames_tensor.mean()) / (frames_tensor.std() + 1e-6)
    frames_tensor = frames_tensor.unsqueeze(0).unsqueeze(2).to(device)
    lengths = torch.tensor([frames_tensor.size(1)], dtype=torch.long)
    log_probs, output_lengths = inference_model.ctc_log_probs(frames_tensor, lengths)
    tokens = ctc_greedy_decode(log_probs.cpu(), output_lengths.cpu(), loaded_blank_index)[0]
    return {'text': tokens_to_text(tokens, loaded_index_to_char), 'frames': frames_array}

video_path = os.path.join('data', 'lips-dataset', 's1', 'swwv8p.mpg')
result = predict_video(video_path)
print('Predicted text:', result['text'])
print('Frames shape:', result['frames'].shape)

## Custom-words training & evaluation
### Train the isolated-word classifier on `data/custom_words/` (8 words: hello, hi, yes, no, good, bad, thanks, bye).

In [3]:
from custom_words_dataset import get_custom_words_dataloaders
from models.LBY.model import LipReadingClassifier

train_loader, test_loader, label_to_index, index_to_label = get_custom_words_dataloaders(
    test_size=0.2, seed=42, batch_size=32
)
num_classes = len(label_to_index)

print(f'Device: {device}')
print(f'Train samples: {len(train_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Classes: {num_classes} {label_to_index}')


Device: cuda
Train samples: 256
Test samples: 64
Classes: 8 {'bad': 0, 'bye': 1, 'good': 2, 'hello': 3, 'hi': 4, 'no': 5, 'thanks': 6, 'yes': 7}


In [4]:
from pathlib import Path

import torch
import torch.nn as nn

model = LipReadingClassifier(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[40, 60], gamma=0.1)
criterion = nn.CrossEntropyLoss()

out_dir = Path('checkpoints')
out_dir.mkdir(parents=True, exist_ok=True)

def run_epoch(loader, training):
    model.train(training)
    total_loss = correct = total = 0
    for frames, labels in loader:
        frames, labels = frames.to(device), labels.to(device)
        with torch.set_grad_enabled(training):
            logits = model(frames)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

epochs = 200
best_accuracy = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    test_loss, test_acc = run_epoch(test_loader, training=False)
    scheduler.step()
    print(f'Epoch {epoch:2d}/{epochs} | train_loss {train_loss:.4f} train_acc {train_acc:.4f} | test_loss {test_loss:.4f} test_acc {test_acc:.4f}')
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'label_to_index': label_to_index, 'test_accuracy': test_acc}, out_dir / 'custom_words.pt')

print(f'Best test accuracy: {best_accuracy:.4f}')


Epoch  1/200 | train_loss 2.1821 train_acc 0.0898 | test_loss 2.0881 test_acc 0.1250
Epoch  2/200 | train_loss 2.1344 train_acc 0.1055 | test_loss 2.0853 test_acc 0.0469
Epoch  3/200 | train_loss 2.1241 train_acc 0.1250 | test_loss 2.0840 test_acc 0.1250
Epoch  4/200 | train_loss 2.0947 train_acc 0.1172 | test_loss 2.0724 test_acc 0.1562
Epoch  5/200 | train_loss 2.0784 train_acc 0.1250 | test_loss 2.0533 test_acc 0.1250
Epoch  6/200 | train_loss 2.0531 train_acc 0.1367 | test_loss 2.0219 test_acc 0.2344
Epoch  7/200 | train_loss 1.9954 train_acc 0.2070 | test_loss 1.9732 test_acc 0.2031
Epoch  8/200 | train_loss 1.9545 train_acc 0.2227 | test_loss 1.8997 test_acc 0.2344
Epoch  9/200 | train_loss 1.8890 train_acc 0.1992 | test_loss 1.8067 test_acc 0.2812
Epoch 10/200 | train_loss 1.7582 train_acc 0.2695 | test_loss 1.7128 test_acc 0.2656
Epoch 11/200 | train_loss 1.6458 train_acc 0.2578 | test_loss 1.5801 test_acc 0.2812
Epoch 12/200 | train_loss 1.6020 train_acc 0.2734 | test_loss 1.5

In [5]:
import numpy as np

checkpoint = torch.load('checkpoints/custom_words.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for frames, labels in test_loader:
        frames, labels = frames.to(device), labels.to(device)
        all_preds.append(model(frames).argmax(dim=1).cpu())
        all_targets.append(labels.cpu())
preds = torch.cat(all_preds).numpy()
targets = torch.cat(all_targets).numpy()

print(f'Overall test accuracy: {float((preds == targets).mean()):.4f}')
print('\nPer-word accuracy:')
for word, idx in label_to_index.items():
    mask = targets == idx
    if mask.sum():
        print(f'  {word:>8}: {float((preds[mask] == idx).mean()):.4f} ({int(mask.sum())} samples)')

print('\nConfusion matrix (rows=truth, cols=prediction):')
conf = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(targets, preds):
    conf[t, p] += 1
print('        ' + ' '.join(f'{index_to_label[i][:4]:>4}' for i in range(num_classes)))
for i in range(num_classes):
    print(f'{index_to_label[i]:>8} ' + ' '.join(f'{v:4d}' for v in conf[i]))


Overall test accuracy: 0.6875

Per-word accuracy:
       bad: 0.0000 (8 samples)
       bye: 0.0000 (8 samples)
      good: 1.0000 (8 samples)
     hello: 1.0000 (8 samples)
        hi: 1.0000 (8 samples)
        no: 1.0000 (8 samples)
    thanks: 0.6250 (8 samples)
       yes: 0.8750 (8 samples)

Confusion matrix (rows=truth, cols=prediction):
         bad  bye good hell   hi   no than  yes
     bad    0    0    0    0    1    0    6    1
     bye    0    0    0    0    8    0    0    0
    good    0    0    8    0    0    0    0    0
   hello    0    0    0    8    0    0    0    0
      hi    0    0    0    0    8    0    0    0
      no    0    0    0    0    0    8    0    0
  thanks    0    0    0    0    1    0    5    2
     yes    0    0    0    0    0    0    1    7
